In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas

from tsfm_lens.chronos2.circuitlens import CircuitLensChronos2
from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.metrics import compute_metrics
from tsfm_lens.utils import apply_custom_style, load_dyst_samples, set_seed


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")


In [ ]:
DEFAULT_CONTEXT_LENGTH = 2048

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "ablations_chronos2", "examples")
os.makedirs(plot_save_dir, exist_ok=True)


### Load Pipeline

In [ ]:
model_id = "amazon/chronos-2"  # replace with desired Chronos-2 checkpoint
context_length = DEFAULT_CONTEXT_LENGTH

pipeline = CircuitLensChronos2(
    model_name=model_id,
    device=device,
)


In [ ]:
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads
print(f"num_layers: {num_layers}, num_heads: {num_heads}")


In [ ]:
model_param_memory_mb = sum(p.numel() * p.element_size() for p in pipeline.model.parameters()) / (1024 * 1024)
print(f"Model parameter memory: {model_param_memory_mb:.2f} MB")


### Load Data

In [ ]:
split_name = "gift-eval"
subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Thomas"

    context_length_data = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length_data

    sample_idx = 0
    selected_dim = [0, 1, 2]

    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :])
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context_target = dyst_coords[:, context_start_time:context_end_time]

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]

elif split_name == "gift-eval":
    from itertools import islice

    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "LOOP_SEATTLE/H"
    to_univariate, term = False, "medium"
    selected_sample_idx, selected_dim = 1, [0, 1, 2]

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)
    prediction_length = dataset.test_data.prediction_length

    context_entry = next(islice(dataset.test_data.input, selected_sample_idx, None))
    future_entry = next(islice(dataset.test_data.label, selected_sample_idx, None))
    context_target = context_entry["target"]
    future_target = future_entry["target"]

    if context_target.ndim == 1:
        context_target, future_target = context_target[None, :], future_target[None, :]
    else:
        context_target, future_target = context_target[selected_dim], future_target[selected_dim]

    orig_context_len = context_target.shape[1]
    truncate_offset = max(0, orig_context_len - DEFAULT_CONTEXT_LENGTH)
    context_target = context_target[:, -DEFAULT_CONTEXT_LENGTH:]
    context_start = context_entry["start"] + truncate_offset

    num_variates = context_target.shape[0]
    fig, axes = plt.subplots(num_variates, 1, figsize=(5, 2 * num_variates), squeeze=False)
    for dim, ax in enumerate(axes.flat):
        to_pandas({"target": context_target[dim], "start": context_start}).plot(
            color="black", linewidth=1, ax=ax, label="Context"
        )
        to_pandas({"target": future_target[dim], "start": future_entry["start"]}).plot(
            color="tab:blue", linewidth=1, ax=ax, label="Ground Truth"
        )
        ax.grid(which="both")
        ax.set_title(f"Dim {dim}")
        ax.legend()
    plt.tight_layout()

    context = torch.tensor(context_target)
    future_vals = torch.tensor(future_target)
    num_nans = np.isnan(context_target).sum() + np.isnan(future_target).sum()
    print(f"Context: {context.shape}, Future: {future_vals.shape}, NaNs: {num_nans}")

# reshape to (batch, time, dim) for pipeline
context = context.transpose(0, 1).unsqueeze(0)
future_vals = future_vals.transpose(0, 1).unsqueeze(0)
context = context.to(device)
future_vals = future_vals.to(device)
print(f"Context reshaped for Chronos2: {context.shape}, Future: {future_vals.shape}")


### Ablations

In [ ]:
pipeline.remove_all_hooks()

chosen_layers = list(range(min(3, pipeline.num_layers)))
heads_to_ablate = [(layer, head) for layer in chosen_layers for head in range(min(2, pipeline.num_heads))]
for layer in chosen_layers:
    heads_in_layer = [h for l, h in heads_to_ablate if l == layer]
    print(f"Layer {layer}: {len(heads_in_layer)} heads")

pipeline.add_ablation_hooks_explicit(
    ablations_types=["head"],  # type: ignore
    layers_to_ablate_mlp=[],
    heads_to_ablate=heads_to_ablate,
    attention_type="time",
)


In [ ]:
layers_without_heads = list(pipeline.time_head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

if layers_without_heads and layers_without_mlps:
    if layers_without_heads != layers_without_mlps:
        raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
    ablations_summary_str_title = f"Zero-Ablation: Heads and MLPs in Layers {layers_without_heads}"
    ablations_summary_str = f"za_heads_mlps_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str = ablations_summary_str_title = None

print(ablations_summary_str)
print(ablations_summary_str_title)


### Predict and Get Outputs

In [ ]:
rseed = 123
set_seed(rseed)

pred = pipeline.predict(context, prediction_length=prediction_length)


In [ ]:
pred.shape

### Plot Predictions with Ablations

In [ ]:
plot_samples = False

In [ ]:
context_np = context.squeeze(0).cpu().numpy().transpose(1, 0)
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.squeeze(0).cpu().numpy().transpose(1, 0)
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)


In [ ]:
preds = pred.detach().cpu().numpy()
if preds.ndim == 2:
    preds = preds[None, ...]
elif preds.ndim == 3:
    preds = preds.transpose(0, 2, 1)  # (batch, horizon, dim) -> (batch, dim, horizon)

if preds.shape[0] == 1:
    preds = preds[0]

num_variates = preds.shape[0]
if preds.ndim == 2:
    preds = preds[:, None, :]
num_samples = preds.shape[1]

fig, axs = plt.subplots(num_variates, 1, figsize=(6, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(
        context_timesteps[-min(512, len(context_timesteps)) :],
        context_np[dim][-min(512, len(context_timesteps)) :],
        "k-",
        linewidth=1,
        label="Context",
    )
    ax.plot(future_timesteps, future_vals_np[dim], "k--", linewidth=1, label="Ground Truth", zorder=10)
    if plot_samples and num_samples > 1:
        for i in range(num_samples):
            ax.plot(
                future_timesteps, preds[dim, i], linewidth=1, color=DEFAULT_COLORS[i % len(DEFAULT_COLORS)], alpha=0.2
            )
    ax.plot(future_timesteps, np.median(preds, axis=1)[dim], color="tab:blue", linewidth=1, label="Median", zorder=11)
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    ax.legend(loc="lower left", frameon=True)

plt.tight_layout()


### Plot Original Predictions

In [ ]:
pipeline.remove_all_hooks()
set_seed(rseed)

pred_original = pipeline.predict(context, prediction_length=prediction_length)


In [ ]:
preds_original = pred_original.detach().cpu().numpy()
if preds_original.ndim == 2:
    preds_original = preds_original[None, ...]
elif preds_original.ndim == 3:
    preds_original = preds_original.transpose(0, 2, 1)
if preds_original.shape[0] == 1:
    preds_original = preds_original[0]
if preds_original.ndim == 2:
    preds_original = preds_original[:, None, :]

num_variates = preds_original.shape[0]
num_samples = preds_original.shape[1]

fig, axs = plt.subplots(num_variates, 1, figsize=(6, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(
        context_timesteps[-min(512, len(context_timesteps)) :],
        context_np[dim][-min(512, len(context_timesteps)) :],
        "k-",
        linewidth=1,
        label="Context",
    )
    ax.plot(future_timesteps, future_vals_np[dim], "k--", label="Ground Truth", linewidth=1, zorder=10)
    if plot_samples and num_samples > 1:
        for i in range(num_samples):
            ax.plot(
                future_timesteps,
                preds_original[dim, i],
                linewidth=1,
                color=DEFAULT_COLORS[i % len(DEFAULT_COLORS)],
                alpha=0.2,
            )
    ax.plot(
        future_timesteps,
        np.median(preds_original, axis=1)[dim],
        color="tab:blue",
        linewidth=1,
        label="Median",
        zorder=11,
    )
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    ax.legend(loc="lower left", frameon=True)

plt.tight_layout()


### Compute Metrics

In [ ]:
med_pred = np.median(preds, axis=1)
med_orig = np.median(preds_original, axis=1)
print(f"med_pred shape: {med_pred.shape}")
print(f"med_orig shape: {med_orig.shape}")


In [ ]:
pred_horizon_lst = [64]
for pred_horizon in pred_horizon_lst:
    for dim in range(med_pred.shape[0]):
        print(f"Computing metrics for prediction horizon {pred_horizon} and dimension {dim}")
        metrics_combined = compute_metrics(med_orig[dim, :pred_horizon], med_pred[dim, :pred_horizon])
        spearman_corr = metrics_combined["spearman"]
        print(f"Spearman correlation: {spearman_corr}")
